In [ ]:
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

In [ ]:
# --- Paths ---
DATA_DIR = Path("../data/long_test")
LIGHT_CSV_PATH = DATA_DIR / "light_intensity.csv"   # photostimulation signal logged by Bonsai
OUTPUT_CSV_PATH = DATA_DIR / "fluorescence_intensity.csv"  # per-frame averages saved here

# --- Acquisition rates ---
TIFF_RATE = 33.32   # Hz — Hamamatsu camera frame rate
LIGHT_RATE = 60     # Hz — light intensity logging rate

In [ ]:
# Verify all required inputs exist before doing any heavy work
assert DATA_DIR.is_dir(), f"Data directory not found: {DATA_DIR.resolve()}"
assert LIGHT_CSV_PATH.is_file(), f"Light intensity CSV not found: {LIGHT_CSV_PATH.resolve()}"

tiff_candidates = sorted(DATA_DIR.glob("*.tif")) + sorted(DATA_DIR.glob("*.tiff"))
assert len(tiff_candidates) > 0, f"No TIFF files found in {DATA_DIR.resolve()}"

print("All files found:")
print(f"  DATA_DIR:   {DATA_DIR.resolve()}")
print(f"  Light CSV:  {LIGHT_CSV_PATH.resolve()}")
print(f"  TIFFs ({len(tiff_candidates)}): {[f.name for f in tiff_candidates]}")

In [ ]:
# Each TIFF (single or multi-page) is read page-by-page and stacked into a 3-D array.
# Multiple files are concatenated in sorted filename order (Hamamatsu splits at ~4 GB).
all_frames = []
for tiff_path in tqdm(tiff_candidates, desc="TIFF files"):
    with tifffile.TiffFile(tiff_path) as tif:
        n_pages = len(tif.pages)
        stack = np.stack(
            [tif.pages[i].asarray()
             for i in tqdm(range(n_pages), desc=tiff_path.name, leave=False)],
            axis=0,
        )
    all_frames.append(stack)

# Final array shape: (total_frames, height, width)
images = np.concatenate(all_frames, axis=0)
n_frames = images.shape[0]
print(f"Image stack shape: {images.shape}  ({n_frames} frames total)")

In [ ]:
# Average all pixels in each frame → one scalar per frame
avg_intensity = images.mean(axis=(1, 2))

fluorescence_df = pd.DataFrame({
    "frame": np.arange(n_frames),
    "time_s": np.arange(n_frames) / TIFF_RATE,
    "avg_intensity": avg_intensity,
})

fluorescence_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Saved {len(fluorescence_df)} rows to {OUTPUT_CSV_PATH}")
fluorescence_df.head()

### 2. Read light intensity CSV

In [ ]:
light_df = pd.read_csv(LIGHT_CSV_PATH)

first_counter = light_df["FrameCounter"].iloc[0]
last_counter  = light_df["FrameCounter"].iloc[-1]
expected_frames = last_counter - first_counter + 1
missing_frames  = expected_frames - len(light_df)

# Use the frame counter to build an absolute time axis so gaps appear at their true position
light_df["time_s"] = (light_df["FrameCounter"] - first_counter) / LIGHT_RATE

print(f"Light intensity: {len(light_df)} recorded samples")
print(f"  Frame counter span : {first_counter} → {last_counter}  ({expected_frames} expected)")
print(f"  Missing frames     : {missing_frames}  ({100 * missing_frames / expected_frames:.1f}%)")
light_df.head()

### 3. Plot both signals on a shared time axis

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(fluorescence_df["time_s"], fluorescence_df["avg_intensity"],
         color="steelblue", linewidth=0.8)
ax1.set_ylabel("Average intensity (a.u.)")
ax1.set_title(f"Fluorescence — TIFF average intensity ({TIFF_RATE} Hz)")
ax1.grid(True, alpha=0.3)

ax2.plot(light_df["time_s"], light_df["FrameIntensity"],
         color="darkorange", linewidth=0.8)
ax2.set_ylabel("Light intensity (a.u.)")
ax2.set_xlabel("Time (s)")
ax2.set_title(f"Light intensity ({LIGHT_RATE} Hz)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(DATA_DIR / "intensity_plot.png", dpi=150)
plt.show()

In [ ]:
def plot_window(title, t_start, t_end):
    """Plot fluorescence and light intensity for a given time window."""
    fluo  = fluorescence_df[fluorescence_df["time_s"].between(t_start, t_end)]
    light = light_df[light_df["time_s"].between(t_start, t_end)]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    ax1.plot(fluo["time_s"], fluo["avg_intensity"], color="steelblue", linewidth=0.8)
    ax1.set_ylabel("Average intensity (a.u.)")
    ax1.set_title(f"Fluorescence ({TIFF_RATE} Hz) — {title}")
    ax1.grid(True, alpha=0.3)

    ax2.plot(light["time_s"], light["FrameIntensity"], color="darkorange", linewidth=0.8)
    ax2.set_ylabel("Light intensity (a.u.)")
    ax2.set_xlabel("Time (s)")
    ax2.set_title(f"Light intensity ({LIGHT_RATE} Hz) — {title}")
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    fname = title.lower().replace(" ", "_") + ".png"
    plt.savefig(DATA_DIR / fname, dpi=150)
    plt.show()

# Use the later of the two signal end times so neither is clipped
t_max = max(fluorescence_df["time_s"].max(), light_df["time_s"].max())

plot_window("first 30s", 0, 30)
plot_window("last 30s", t_max - 30, t_max)